# Scale study v2 — multi-step (2-hop): solving vs late-stage refinement across size

`Runtime -> Run all`. Loads Qwen2.5 0.5B->7B and on a **2-hop** task (city -> country -> capital)
tracks, per layer, whether the **bridge country** leads (mid-stack) then the **capital** overtakes
(late). Prints a VERDICT: does solving grow, does the bridge form mid-stack, does the **hop-2
completion** (late refinement) grow with scale? Defensive: a model failing is skipped.

## 1 · Environment self-heal (run first)
If a prior install corrupted numpy, this says so — fix is **Runtime -> Disconnect and delete runtime**, reconnect, Run all.

In [ ]:
import sys, subprocess, os
try:
    import numpy, scipy.sparse, matplotlib  # noqa
    print('env OK — numpy', numpy.__version__)
except Exception as e:
    raise SystemExit('BROKEN ENV: '+repr(e)+'\n>>> Runtime -> Disconnect and delete runtime -> reconnect -> Run all.')


## 2 · Get the code + deps (no downgrades)

In [ ]:
import os
os.chdir('/content')
REPO='https://github.com/sinha-k-prat/small_fable.git'; BR='residual-orrery'
if not os.path.isdir('small_fable'):
    !git clone -q -b $BR $REPO
os.chdir('/content/small_fable')
!git fetch -q origin $BR && git reset --hard -q origin/$BR
!pip install -q bitsandbytes accelerate 'transformers>=4.44'
import transformers, torch
print('transformers', transformers.__version__, '| cuda', torch.cuda.is_available())


## 3 · Run the study (the whole thing)
Edit `--models` to drop 7B if you are tight on GPU/disk (e.g. `--models 0.5B 1.5B 3B`).
Bigger `--n` = more examples = steadier numbers (slower).

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (slow on CPU)')
# 2-hop multi-step study: bridge(country) forms mid -> capital overtakes late; does it scale?
!python tools/scale_study2.py --models 0.5B 1.5B 3B 7B --out scale2
from IPython.display import Image, display
display(Image(filename='scale2.png'))


## 4 · (optional) download the raw numbers

In [ ]:
import json, os
print(json.dumps(json.load(open('scale2.json')), indent=2)[:3000])
try:
    from google.colab import files; files.download('scale2.json'); files.download('scale2.png')
except Exception:
    print('saved at', os.path.abspath('scale2.json'))
